In [1]:
%pip install flask

Note: you may need to restart the kernel to use updated packages.


### Dependencies
Flask provides the REST API. The demonstration uses Flask's test client,
which sends requests to the application within the notebook without
starting a long-running web server.

In [2]:
%%writefile network.py
"""Peer registration for the Assignment 8 classroom blockchain."""

from urllib.parse import urlparse


def normalise_address(address):
    """Convert a peer address into a consistent HTTP/HTTPS base URL."""
    if not isinstance(address, str) or not address.strip():
        raise ValueError("Each node address must be a non-empty string.")

    address = address.strip()
    if "://" not in address:
        address = "http://" + address

    parsed = urlparse(address)

    if parsed.scheme not in {"http", "https"} or not parsed.hostname:
        raise ValueError("Node addresses must use HTTP or HTTPS.")

    if parsed.username or parsed.password:
        raise ValueError("Credentials are not allowed in node addresses.")

    if parsed.path not in {"", "/"} or parsed.query or parsed.fragment:
        raise ValueError("Use a base URL without a path, query or fragment.")

    try:
        port = parsed.port
    except ValueError:
        raise ValueError("Invalid node port.") from None

    if port is not None and not 1 <= port <= 65535:
        raise ValueError("Node port must be between 1 and 65535.")

    host = parsed.hostname.lower()
    if ":" in host:
        host = f"[{host}]"  # Preserve IPv6 URL syntax.

    suffix = f":{port}" if port is not None else ""
    return f"{parsed.scheme}://{host}{suffix}"


class NodeDirectory:
    """Store unique peer URLs; registration does not establish a connection."""

    def __init__(self):
        self.nodes = set()

    def register_many(self, addresses):
        if not isinstance(addresses, list) or not addresses:
            raise ValueError("'nodes' must be a non-empty list.")

        # Validate the complete request before changing the peer directory.
        normalised = {normalise_address(item) for item in addresses}
        self.nodes.update(normalised)
        return self.list_nodes()

    def list_nodes(self):
        return sorted(self.nodes)

Writing network.py


### Peer directory — network.py
The peer directory validates and normalises peer URLs, removes duplicates
and returns a sorted list.

Registration records a peer's address only. It does not implement Bitcoin
peer discovery, transaction relay or chain synchronisation.

In [3]:
%%writefile explorer.py
"""Read-only explorer views for the same ledger used by the API."""

import json


def format_chain(chain, limit=5):
    """Show recent blocks and their transactions, newest first."""
    recent = list(reversed(chain[-limit:]))
    lines = [
        f"Showing {len(recent)} of {len(chain)} blocks (newest first)",
        ""
    ]

    for block in recent:
        lines.extend([
            f"Block index: {block['index']}",
            f"Hash: {block['hash']}",
            f"Previous hash: {block['previous_hash']}",
            f"Timestamp: {block['timestamp']}",
            f"Nonce: {block['nonce']}",
            f"Transactions: {len(block['transactions'])}"
        ])

        for transaction in block["transactions"]:
            lines.append("  " + json.dumps(transaction, sort_keys=True))

        lines.append("")

    return "\n".join(lines)


EXPLORER_HTML = """
<!doctype html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <title>Assignment 8 Blockchain Explorer</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 30px;
            background: #f3f6fa;
            color: #183048;
        }
        article {
            background: white;
            padding: 20px;
            margin-bottom: 20px;
            border-left: 5px solid #167d9a;
            border-radius: 6px;
        }
        code { overflow-wrap: anywhere; }
        table { width: 100%; border-collapse: collapse; }
        th, td {
            padding: 10px;
            text-align: left;
            border-bottom: 1px solid #dde4eb;
            overflow-wrap: anywhere;
        }
        .table-wrap { overflow-x: auto; }
    </style>
</head>
<body>
    <h1>Assignment 8 Blockchain Explorer</h1>
    <p>Chain length: {{ length }} | Pending transactions: {{ pending }}</p>
    <p>Recent blocks appear first. Amounts are classroom ledger units.</p>

    {% for block in blocks %}
    <article>
        <h2>Block {{ block.index }}</h2>
        <p><strong>Hash:</strong> <code>{{ block.hash }}</code></p>
        <p><strong>Previous hash:</strong>
           <code>{{ block.previous_hash }}</code></p>
        <p><strong>Timestamp:</strong> {{ block.timestamp }}
           | <strong>Nonce:</strong> {{ block.nonce }}</p>

        <div class="table-wrap">
        <table>
            <tr>
                <th>Transaction ID</th>
                <th>Sender</th>
                <th>Recipient</th>
                <th>Amount</th>
                <th>Type</th>
            </tr>
            {% for tx in block.transactions %}
            <tr>
                <td><code>{{ tx.id }}</code></td>
                <td>{{ tx.sender }}</td>
                <td>{{ tx.recipient }}</td>
                <td>{{ tx.amount }}</td>
                <td>{{ tx.type }}</td>
            </tr>
            {% else %}
            <tr><td colspan="5">No transactions: genesis block.</td></tr>
            {% endfor %}
        </table>
        </div>
    </article>
    {% endfor %}
</body>
</html>
"""

Writing explorer.py


### Explorer — explorer.py
The explorer displays recent blocks and the transactions included in each
block. Both the HTML and text views read the API's existing ledger rather
than maintaining a separate chain.

The genesis block contains no payments. Mining rewards are labelled
separately from payment transactions.

In [4]:
%%writefile api.py
"""Localhost-only teaching API. State is held in memory."""

import hashlib
import json
import math
import time
from copy import deepcopy
from threading import RLock
from uuid import uuid4

from flask import Flask, jsonify, request, render_template_string

from network import NodeDirectory
from explorer import EXPLORER_HTML, format_chain


DIFFICULTY = 2
MINING_REWARD = 1.0


def block_hash(block):
    """Hash every block field except the stored hash itself."""
    payload = {key: value for key, value in block.items() if key != "hash"}
    encoded = json.dumps(
        payload, sort_keys=True, separators=(",", ":"), allow_nan=False
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


class Blockchain:
    def __init__(self):
        self.chain = []
        self.mempool = []
        self.payments = {}
        self.nodes = NodeDirectory()
        self.lock = RLock()

        # Fixed genesis fields permit a reproducible genesis block.
        genesis = {
            "index": 0,
            "timestamp": 0,
            "transactions": [],
            "previous_hash": "0" * 64,
            "nonce": 0
        }
        self.chain.append(self.proof_of_work(genesis))

    @staticmethod
    def proof_of_work(block):
        while True:
            candidate_hash = block_hash(block)
            if candidate_hash.startswith("0" * DIFFICULTY):
                block["hash"] = candidate_hash
                return block
            block["nonce"] += 1

    def new_transaction(self, body):
        sender = body.get("sender")
        recipient = body.get("recipient")
        amount = body.get("amount")
        tx_id = body.get("tx_id")

        if not isinstance(sender, str) or not sender.strip():
            raise ValueError("'sender' must be a non-empty string.")
        if not isinstance(recipient, str) or not recipient.strip():
            raise ValueError("'recipient' must be a non-empty string.")
        if sender.strip() == "0":
            raise ValueError("Sender '0' is reserved for mining rewards.")

        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            raise ValueError("'amount' must be a positive finite number.")

        try:
            amount = float(amount)
        except (ValueError, OverflowError):
            raise ValueError("'amount' must be a positive finite number.") from None

        if not math.isfinite(amount) or amount <= 0:
            raise ValueError("'amount' must be a positive finite number.")

        if tx_id is not None:
            if not isinstance(tx_id, str) or not tx_id.strip():
                raise ValueError("'tx_id' must be a non-empty string.")
            tx_id = tx_id.strip()
        else:
            tx_id = str(uuid4())

        sender, recipient = sender.strip(), recipient.strip()

        with self.lock:
            if tx_id in self.payments:
                existing = self.payments[tx_id]
                same_payment = (
                    existing["sender"] == sender
                    and existing["recipient"] == recipient
                    and existing["amount"] == amount
                )
                if not same_payment:
                    return None, False, True
                return deepcopy(existing), False, False

            transaction = {
                "id": tx_id,
                "sender": sender,
                "recipient": recipient,
                "amount": amount,
                "timestamp": time.time(),
                "type": "payment"
            }
            self.mempool.append(transaction)
            self.payments[tx_id] = transaction
            return deepcopy(transaction), True, False

    def transaction_status(self, tx_id):
        for block in self.chain:
            if any(tx["id"] == tx_id for tx in block["transactions"]):
                return "included", block["index"]
        return "pending", None

    def mine_block(self):
        with self.lock:
            reward = {
                "id": str(uuid4()),
                "sender": "0",
                "recipient": "classroom-miner",
                "amount": MINING_REWARD,
                "timestamp": time.time(),
                "type": "coinbase"
            }
            block = {
                "index": len(self.chain),
                "timestamp": time.time(),
                "transactions": deepcopy(self.mempool) + [reward],
                "previous_hash": self.chain[-1]["hash"],
                "nonce": 0
            }
            block = self.proof_of_work(block)
            self.chain.append(block)
            self.mempool.clear()
            return deepcopy(block)

    def verify_chain(self):
        """Verify indices, hash commitments, PoW and predecessor links."""
        for index, block in enumerate(self.chain):
            if block["index"] != index:
                return False
            if block_hash(block) != block["hash"]:
                return False
            if not block["hash"].startswith("0" * DIFFICULTY):
                return False
            expected_previous = (
                "0" * 64 if index == 0 else self.chain[index - 1]["hash"]
            )
            if block["previous_hash"] != expected_previous:
                return False
        return bool(self.chain)


def create_app():
    app = Flask(__name__)
    ledger = Blockchain()
    app.config["ledger"] = ledger

    def json_body():
        body = request.get_json(silent=True)
        if not isinstance(body, dict):
            raise ValueError("Provide a JSON object with Content-Type application/json.")
        return body

    @app.get("/")
    def index():
        return jsonify({
            "service": "Assignment 8 classroom blockchain",
            "endpoints": [
                "GET /chain",
                "POST /transactions/new",
                "POST /mine",
                "GET /nodes",
                "POST /nodes/register",
                "GET /explorer",
                "GET /explorer.txt"
            ]
        })

    @app.get("/chain")
    def get_chain():
        with ledger.lock:
            return jsonify({
                "chain": deepcopy(ledger.chain),
                "length": len(ledger.chain),
                "pending_transactions": len(ledger.mempool),
                "valid": ledger.verify_chain()
            })

    @app.post("/transactions/new")
    def new_transaction():
        try:
            with ledger.lock:
                tx, created, conflict = ledger.new_transaction(json_body())
                if conflict:
                    return jsonify({
                        "error": "This tx_id already identifies a different payment."
                    }), 409

                status, block_index = ledger.transaction_status(tx["id"])
                return jsonify({
                    "message": "Payment accepted." if created else "Existing payment returned.",
                    "transaction": tx,
                    "status": status,
                    "block_index": block_index,
                    "next_block_index": len(ledger.chain),
                    "duplicate": not created
                }), 201 if created else 200
        except ValueError as error:
            return jsonify({"error": str(error)}), 400

    @app.post("/mine")
    def mine():
        block = ledger.mine_block()
        return jsonify({
            "message": "Block mined.",
            "block": block,
            "length": block["index"] + 1
        }), 200

    @app.get("/nodes")
    def get_nodes():
        with ledger.lock:
            return jsonify({"nodes": ledger.nodes.list_nodes()})

    @app.post("/nodes/register")
    def register_nodes():
        try:
            body = json_body()
            with ledger.lock:
                nodes = ledger.nodes.register_many(body.get("nodes"))
            return jsonify({
                "message": "Peer directory updated.",
                "nodes": nodes
            }), 201
        except ValueError as error:
            return jsonify({"error": str(error)}), 400

    @app.get("/explorer")
    def explorer_html():
        with ledger.lock:
            return render_template_string(
                EXPLORER_HTML,
                blocks=deepcopy(list(reversed(ledger.chain[-5:]))),
                length=len(ledger.chain),
                pending=len(ledger.mempool)
            )

    @app.get("/explorer.txt")
    def explorer_text():
        with ledger.lock:
            return app.response_class(
                format_chain(ledger.chain),
                mimetype="text/plain"
            )

    return app


if __name__ == "__main__":
    create_app().run(host="127.0.0.1", port=5000, debug=False)

Writing api.py


### REST API — api.py
The API exposes the required chain, transaction, mining and peer endpoints.
Payments enter the mempool before being included in a mined block.

Each block commits to its contents through SHA-256 and links to the previous
block. The classroom proof-of-work requires two leading hexadecimal zeros.
The chain verifier checks block indices, hashes, proof-of-work and links.

A supplied tx_id supports idempotent payment retries within the running
application. Repeating the same payment returns HTTP 200 without adding
another transaction. Reusing its ID for different payment details returns
HTTP 409.

This teaching ledger does not verify signatures, balances or spending
authority. Its amounts are illustrative units, not actual rand transfers.
State and duplicate tracking are lost when the application restarts.

In [5]:
# Reload local modules so rerunning the notebook uses the latest file contents.
import importlib
import json

import network
import explorer
import api

importlib.reload(network)
importlib.reload(explorer)
importlib.reload(api)

app = api.create_app()
client = app.test_client()


def show_response(label, response):
    print(f"\n{label}")
    print(f"HTTP status: {response.status_code}")
    print(json.dumps(response.get_json(), indent=2))


print("Application ready.")
print("Use this same client for the following demonstration cells.")

Application ready.
Use this same client for the following demonstration cells.


### Initialise the demonstration
A fresh application starts with one genesis block and an empty mempool.
All following requests use the same application instance.

Rerunning this cell resets the demonstration ledger.

In [6]:
initial = client.get("/chain")
show_response("1. INITIAL CHAIN — GET /chain", initial)

assert initial.status_code == 200
assert initial.get_json()["length"] == 1
assert initial.get_json()["pending_transactions"] == 0
assert initial.get_json()["valid"] is True


1. INITIAL CHAIN — GET /chain
HTTP status: 200
{
  "chain": [
    {
      "hash": "00025fc8a03ccdc2b9fce459ba9ae3fb8697d81df8bc362fefafa34760fadddc",
      "index": 0,
      "nonce": 107,
      "previous_hash": "0000000000000000000000000000000000000000000000000000000000000000",
      "timestamp": 0,
      "transactions": []
    }
  ],
  "length": 1,
  "pending_transactions": 0,
  "valid": true
}


### Initial chain
GET /chain returns HTTP 200, one genesis block, no pending transactions
and a valid chain. This establishes the state before submitting a payment.

In [7]:
payment = {
    "sender": "Alice",
    "recipient": "Bob",
    "amount": 10000,
    "tx_id": "A8-PAYMENT-001"
}

submitted = client.post("/transactions/new", json=payment)
show_response("2. SUBMIT PAYMENT — POST /transactions/new", submitted)

pending = client.get("/chain")
show_response("3. CHECK PENDING STATE — GET /chain", pending)

assert submitted.status_code == 201
assert submitted.get_json()["status"] == "pending"
assert pending.get_json()["length"] == 1
assert pending.get_json()["pending_transactions"] == 1


2. SUBMIT PAYMENT — POST /transactions/new
HTTP status: 201
{
  "block_index": null,
  "duplicate": false,
  "message": "Payment accepted.",
  "next_block_index": 1,
  "status": "pending",
  "transaction": {
    "amount": 10000.0,
    "id": "A8-PAYMENT-001",
    "recipient": "Bob",
    "sender": "Alice",
    "timestamp": 1791500719.3998058,
    "type": "payment"
  }
}

3. CHECK PENDING STATE — GET /chain
HTTP status: 200
{
  "chain": [
    {
      "hash": "00025fc8a03ccdc2b9fce459ba9ae3fb8697d81df8bc362fefafa34760fadddc",
      "index": 0,
      "nonce": 107,
      "previous_hash": "0000000000000000000000000000000000000000000000000000000000000000",
      "timestamp": 0,
      "transactions": []
    }
  ],
  "length": 1,
  "pending_transactions": 1,
  "valid": true
}


### Payment submission
The illustrative 10,000-unit payment from Alice to Bob returns HTTP 201.
The chain still contains one block, while the mempool contains one payment.

Acceptance by the API is not block inclusion or final settlement.

In [8]:
retry = client.post("/transactions/new", json=payment)
show_response("4. RETRY SAME PAYMENT — POST /transactions/new", retry)

assert retry.status_code == 200
assert retry.get_json()["duplicate"] is True
assert client.get("/chain").get_json()["pending_transactions"] == 1

print("\nThe retry did not create a second pending payment.")


4. RETRY SAME PAYMENT — POST /transactions/new
HTTP status: 200
{
  "block_index": null,
  "duplicate": true,
  "message": "Existing payment returned.",
  "next_block_index": 1,
  "status": "pending",
  "transaction": {
    "amount": 10000.0,
    "id": "A8-PAYMENT-001",
    "recipient": "Bob",
    "sender": "Alice",
    "timestamp": 1791500719.3998058,
    "type": "payment"
  }
}

The retry did not create a second pending payment.


### Idempotent retry
Repeating the same tx_id and payment details returns the existing payment
with HTTP 200. The mempool remains at one payment.

This prevents duplicate submission when a client retries after an uncertain
response, provided it reuses the same ID within the running application.

In [9]:
mined = client.post("/mine")
show_response("5. MINE BLOCK — POST /mine", mined)

confirmed = client.get("/chain")
show_response("6. VERIFY CHAIN — GET /chain", confirmed)

chain_data = confirmed.get_json()
latest_block = chain_data["chain"][-1]

assert mined.status_code == 200
assert chain_data["length"] == 2
assert chain_data["pending_transactions"] == 0
assert chain_data["valid"] is True

assert any(
    tx["id"] == "A8-PAYMENT-001"
    and tx["sender"] == "Alice"
    and tx["recipient"] == "Bob"
    and tx["amount"] == 10000
    for tx in latest_block["transactions"]
)

assert sum(
    tx["type"] == "coinbase"
    for tx in latest_block["transactions"]
) == 1

print("\nPayment included, mempool cleared and chain verification passed.")


5. MINE BLOCK — POST /mine
HTTP status: 200
{
  "block": {
    "hash": "008589c3987e90744d65c5a1a68d568206d9fe6d0a2f1126c0ad288adacb872c",
    "index": 1,
    "nonce": 284,
    "previous_hash": "00025fc8a03ccdc2b9fce459ba9ae3fb8697d81df8bc362fefafa34760fadddc",
    "timestamp": 1791500865.996383,
    "transactions": [
      {
        "amount": 10000.0,
        "id": "A8-PAYMENT-001",
        "recipient": "Bob",
        "sender": "Alice",
        "timestamp": 1791500719.3998058,
        "type": "payment"
      },
      {
        "amount": 1.0,
        "id": "8a2b9789-2ce7-4739-95a9-c9812f0c7740",
        "recipient": "classroom-miner",
        "sender": "0",
        "timestamp": 1791500865.9963777,
        "type": "coinbase"
      }
    ]
  },
  "length": 2,
  "message": "Block mined."
}

6. VERIFY CHAIN — GET /chain
HTTP status: 200
{
  "chain": [
    {
      "hash": "00025fc8a03ccdc2b9fce459ba9ae3fb8697d81df8bc362fefafa34760fadddc",
      "index": 0,
      "nonce": 107,
      "previo

### Mining and verification
POST /mine includes the pending payment and one classroom mining reward.
The chain grows to two blocks and the mempool becomes empty.

GET /chain verifies the block commitments and links. Inclusion in this
single-node demonstration does not establish network-wide consensus,
irreversible settlement or actual fiat payout.

In [10]:
registered = client.post(
    "/nodes/register",
    json={
        "nodes": [
            "http://127.0.0.1:5001/",
            "127.0.0.1:5002",
            "http://127.0.0.1:5001"
        ]
    }
)
show_response("7. REGISTER PEERS — POST /nodes/register", registered)

peers = client.get("/nodes")
show_response("8. LIST PEERS — GET /nodes", peers)

assert registered.status_code == 201
assert peers.status_code == 200
assert peers.get_json()["nodes"] == [
    "http://127.0.0.1:5001",
    "http://127.0.0.1:5002"
]


7. REGISTER PEERS — POST /nodes/register
HTTP status: 201
{
  "message": "Peer directory updated.",
  "nodes": [
    "http://127.0.0.1:5001",
    "http://127.0.0.1:5002"
  ]
}

8. LIST PEERS — GET /nodes
HTTP status: 200
{
  "nodes": [
    "http://127.0.0.1:5001",
    "http://127.0.0.1:5002"
  ]
}


### Peer registration
Peer registration returns HTTP 201. Addresses are normalised and duplicate
entries are removed, leaving two unique peer URLs.

These entries demonstrate the directory contract; no running peer servers
or successful network connections are implied.

In [11]:
from IPython.display import HTML, display

text_view = client.get("/explorer.txt")
print(text_view.get_data(as_text=True))

html_view = client.get("/explorer")
display(HTML(html_view.get_data(as_text=True)))

assert text_view.status_code == 200
assert html_view.status_code == 200
assert "A8-PAYMENT-001" in html_view.get_data(as_text=True)
assert "Alice" in html_view.get_data(as_text=True)
assert "Bob" in html_view.get_data(as_text=True)

Showing 2 of 2 blocks (newest first)

Block index: 1
Hash: 008589c3987e90744d65c5a1a68d568206d9fe6d0a2f1126c0ad288adacb872c
Previous hash: 00025fc8a03ccdc2b9fce459ba9ae3fb8697d81df8bc362fefafa34760fadddc
Timestamp: 1791500865.996383
Nonce: 284
Transactions: 2
  {"amount": 10000.0, "id": "A8-PAYMENT-001", "recipient": "Bob", "sender": "Alice", "timestamp": 1791500719.3998058, "type": "payment"}
  {"amount": 1.0, "id": "8a2b9789-2ce7-4739-95a9-c9812f0c7740", "recipient": "classroom-miner", "sender": "0", "timestamp": 1791500865.9963777, "type": "coinbase"}

Block index: 0
Hash: 00025fc8a03ccdc2b9fce459ba9ae3fb8697d81df8bc362fefafa34760fadddc
Previous hash: 0000000000000000000000000000000000000000000000000000000000000000
Timestamp: 0
Nonce: 107
Transactions: 0



### Explorer evidence
The explorer displays the mined payment, mining reward and genesis block.
It shows block hashes, predecessor hashes, timestamps, nonces and transaction
details from the same ledger used by GET /chain.

The HTML view provides a readable screenshot for the report.

In [12]:
from copy import deepcopy

invalid_payment = client.post(
    "/transactions/new",
    json={"sender": "Alice", "recipient": "Bob", "amount": -100}
)
show_response("9. INVALID AMOUNT", invalid_payment)
assert invalid_payment.status_code == 400

conflicting_payment = client.post(
    "/transactions/new",
    json={**payment, "amount": 20000}
)
show_response("10. CONFLICTING TRANSACTION ID", conflicting_payment)
assert conflicting_payment.status_code == 409

invalid_peer = client.post(
    "/nodes/register",
    json={"nodes": ["ftp://127.0.0.1:5003"]}
)
show_response("11. INVALID PEER SCHEME", invalid_peer)
assert invalid_peer.status_code == 400

# Check that retry protection also works after mining.
mined_retry = client.post("/transactions/new", json=payment)
assert mined_retry.status_code == 200
assert mined_retry.get_json()["status"] == "included"
assert client.get("/chain").get_json()["pending_transactions"] == 0

# Tamper with an isolated copy; preserve the demonstration application.
tampered_ledger = api.Blockchain()
tampered_ledger.chain = deepcopy(app.config["ledger"].chain)

for tx in tampered_ledger.chain[-1]["transactions"]:
    if tx["id"] == "A8-PAYMENT-001":
        tx["amount"] = 99999

assert tampered_ledger.verify_chain() is False

print("\nTamper detection: PASSED")
print("Original demonstration chain remains valid:",
      app.config["ledger"].verify_chain())
print("\nALL NOTEBOOK DEMONSTRATION CHECKS PASSED.")


9. INVALID AMOUNT
HTTP status: 400
{
  "error": "'amount' must be a positive finite number."
}

10. CONFLICTING TRANSACTION ID
HTTP status: 409
{
  "error": "This tx_id already identifies a different payment."
}

11. INVALID PEER SCHEME
HTTP status: 400
{
  "error": "Node addresses must use HTTP or HTTPS."
}

Tamper detection: PASSED
Original demonstration chain remains valid: True

ALL NOTEBOOK DEMONSTRATION CHECKS PASSED.


### Validation and integrity checks
The demonstration checks that:
- A negative payment amount returns HTTP 400.
- Reusing a transaction ID for a different amount returns HTTP 409.
- An unsupported peer URL scheme returns HTTP 400.
- Retrying an already mined payment does not add another pending payment.
- Altering a transaction in a copied chain causes verification to fail.

Tamper detection checks consistency against stored hashes. It does not
prevent an attacker from recomputing proof-of-work, nor does it replace
distributed consensus or transaction authorisation.

In [13]:
%%writefile README.md
# Assignment 8 — REST API and Networking Lab

## Files
- api.py: Flask routes, in-memory ledger and proof-of-work.
- network.py: peer URL validation and registration.
- explorer.py: text and HTML explorer.
- Notebook: reproducible payment-flow demonstration and checks.

## Installation
python -m pip install flask

## Notebook demonstration
Run the notebook cells in order. The demonstration uses Flask's test_client,
so no external server is needed. Rerunning the initialisation cell resets
the ledger.

## Optional localhost server
From a terminal in the directory containing the three Python files:
python api.py

Open http://127.0.0.1:5000/explorer in a browser.
The server binds to 127.0.0.1:5000 with debug mode disabled.
The terminal server has separate state from the notebook application.
Stop it with Ctrl+C.

## Endpoint contract

| Method | Path | Successful response |
|---|---|---|
| GET | / | 200: service and endpoint list |
| GET | /chain | 200: chain, length, pending_transactions, valid |
| POST | /transactions/new | 201: new payment; 200: identical retry |
| POST | /mine | 200: mined block and chain length |
| GET | /nodes | 200: normalised peer list |
| POST | /nodes/register | 201: updated peer list |
| GET | /explorer | 200: HTML blocks and transactions |
| GET | /explorer.txt | 200: plain-text blocks and transactions |

## Request bodies
POST /transactions/new:
{"sender":"Alice","recipient":"Bob","amount":10000,"tx_id":"A8-PAYMENT-001"}

POST /nodes/register:
{"nodes":["http://127.0.0.1:5001","http://127.0.0.1:5002"]}

POST /mine requires no request body.

## Error responses
Invalid JSON objects, payment fields or peer addresses return 400.
Reusing a tx_id for different payment details returns 409.
Unsupported HTTP methods receive Flask's standard 405 response.

## Payment flow
GET /chain -> POST /transactions/new -> retry the same payment ->
GET /chain -> POST /mine -> GET /chain -> GET /explorer.

## Assumptions and limitations
- Amounts represent classroom ledger units.
- Difficulty is fixed at two leading hexadecimal zeros.
- Each mined block includes one reward of 1 unit.
- Mining an empty mempool creates a reward-only block.
- Duplicate protection requires reuse of tx_id.
- Omitting tx_id creates a new payment ID on each submission.
- State and duplicate tracking are in memory and disappear on restart.
- Peer registration does not connect to peers, relay messages or synchronise.
- Signatures, balances, double-spend prevention and fork choice are not implemented.
- Mining and submission are unauthenticated for the local teaching demo.
- No production payment security, compliance or fiat payout is implied.

Writing README.md


### Documentation
The README describes installation, the payment demonstration, endpoint
contracts, response codes and implementation limitations.

The notebook provides the required scripted payment flow. Its requests
exercise the Flask application internally rather than a live TCP connection.

In [14]:
# Append the graduate-extension demonstration to network.py.
# This cell is safe to rerun: it replaces the previous extension.

from pathlib import Path

network_path = Path("network.py")
marker = "# A8 GRADUATE RELAY EXTENSION"

extension = '''
# A8 GRADUATE RELAY EXTENSION

import hashlib
import json
from copy import deepcopy


def relay_object_id(transaction):
    """Content hash used to identify a simulated transaction object."""
    encoded = json.dumps(
        transaction,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


class RelayNode:
    """In-memory model of address and inventory exchange."""

    def __init__(self, address):
        self.address = normalise_address(address)
        self.directory = NodeDirectory()
        self.objects = {}

    def receive_addr(self, addresses):
        """Model receiving peer addresses; exclude our own address."""
        normalised = [normalise_address(item) for item in addresses]
        others = [item for item in normalised if item != self.address]
        if others:
            self.directory.register_many(others)
        return self.directory.list_nodes()

    def getaddr(self):
        """Return known addresses to a simulated requesting peer."""
        return self.directory.list_nodes()

    def store_transaction(self, transaction):
        """Store a transaction and identify it by its content hash."""
        object_id = relay_object_id(transaction)
        self.objects[object_id] = deepcopy(transaction)
        return object_id

    def inv(self):
        """Announce identifiers, rather than full transaction objects."""
        return [
            {"type": "tx", "hash": object_id}
            for object_id in sorted(self.objects)
        ]

    def missing_inventory(self, inventory):
        """Select announced transactions not already held locally."""
        return [
            deepcopy(item)
            for item in inventory
            if item.get("type") == "tx"
            and item.get("hash") not in self.objects
        ]

    def getdata(self, requested):
        """Return the requested objects available at this node."""
        return [
            {
                "type": "tx",
                "hash": item["hash"],
                "transaction": deepcopy(self.objects[item["hash"]])
            }
            for item in requested
            if item.get("type") == "tx"
            and item.get("hash") in self.objects
        ]

    def receive_data(self, messages):
        """Check content hashes before storing received objects."""
        # Check the complete batch before changing local state.
        checked = []
        for message in messages:
            if message.get("type") != "tx":
                raise ValueError("Unsupported relay object type.")

            transaction = message["transaction"]
            expected_hash = relay_object_id(transaction)

            if expected_hash != message.get("hash"):
                raise ValueError("Relay object hash mismatch.")

            checked.append((expected_hash, deepcopy(transaction)))

        for object_id, transaction in checked:
            self.objects[object_id] = transaction

        return len(checked)
'''

original = network_path.read_text(encoding="utf-8")
original = original.split(marker)[0].rstrip()

network_path.write_text(
    original + "\n\n" + extension.strip() + "\n",
    encoding="utf-8"
)

print("Relay simulation added to network.py.")

Relay simulation added to network.py.


### Part B: address and inventory relay extension
The extension models peer address exchange and transaction inventory relay
between in-memory nodes.

getaddr returns known peer addresses, while receive_addr records received
addresses. inv announces transaction content hashes; getdata returns only
requested objects. Receivers recompute each object's hash before storing it.

This simulation checks content integrity only. It does not verify signatures,
balances or Bitcoin transaction validity, establish network connections,
or implement Bitcoin's binary message format.

In [15]:
import importlib
from copy import deepcopy

importlib.reload(network)

# Three simulated nodes; these are objects, not running HTTP servers.
node_a = network.RelayNode("http://127.0.0.1:5000")
node_b = network.RelayNode("http://127.0.0.1:5001")
node_c = network.RelayNode("http://127.0.0.1:5002")

# 1. ADDRESS EXCHANGE: A knows B and C; B requests A's known addresses.
node_a.receive_addr([node_b.address, node_c.address])
advertised_addresses = node_a.getaddr()
node_b.receive_addr(advertised_addresses)

# B already knows the peer from which it requested the addresses.
node_b.receive_addr([node_a.address])

print("1. GETADDR / ADDR")
print("A advertises:", advertised_addresses)
print("B records:", node_b.directory.list_nodes())

assert node_b.directory.list_nodes() == [
    node_a.address,
    node_c.address
]

# 2. A stores and announces a simulated payment object.
relay_payment = {
    "sender": "Alice",
    "recipient": "Bob",
    "amount": 10000,
    "tx_id": "A8-RELAY-001"
}

object_id = node_a.store_transaction(relay_payment)
announcement = node_a.inv()

print("\n2. INV")
print("A announces:", announcement)

# 3. B requests only objects it does not already hold.
requested = node_b.missing_inventory(announcement)

print("\n3. GETDATA")
print("B requests:", requested)

assert len(requested) == 1

# 4. A supplies the requested object; B checks and stores it.
messages = node_a.getdata(requested)
received_count = node_b.receive_data(messages)

print("\n4. DATA DELIVERY AND INTEGRITY CHECK")
print("Objects received:", received_count)
print("B stores:", node_b.objects[object_id])

assert node_b.objects[object_id] == relay_payment

# 5. A repeated announcement requires no second download.
repeat_request = node_b.missing_inventory(node_a.inv())

print("\n5. REPEATED INVENTORY")
print("Objects requested again:", len(repeat_request))

assert repeat_request == []

# 6. B can announce the object onwards to C.
c_requested = node_c.missing_inventory(node_b.inv())
node_c.receive_data(node_b.getdata(c_requested))

assert node_c.objects[object_id] == relay_payment
print("\n6. ONWARD RELAY")
print("C received the same transaction:", True)

# 7. Altered data must fail its advertised content-hash check.
altered_messages = deepcopy(messages)
altered_messages[0]["transaction"]["amount"] = 99999

try:
    node_c.receive_data(altered_messages)
except ValueError as error:
    print("\n7. ALTERED OBJECT REJECTED")
    print(error)
else:
    raise AssertionError("Altered relay data was incorrectly accepted.")

assert node_c.objects[object_id] == relay_payment

print("\nALL RELAY SIMULATION CHECKS PASSED.")

1. GETADDR / ADDR
A advertises: ['http://127.0.0.1:5001', 'http://127.0.0.1:5002']
B records: ['http://127.0.0.1:5000', 'http://127.0.0.1:5002']

2. INV
A announces: [{'type': 'tx', 'hash': '4e6ebdad81c43a8280a6e0bea4745e65b18f6f2abb38f5a039d44f5ea30c5516'}]

3. GETDATA
B requests: [{'type': 'tx', 'hash': '4e6ebdad81c43a8280a6e0bea4745e65b18f6f2abb38f5a039d44f5ea30c5516'}]

4. DATA DELIVERY AND INTEGRITY CHECK
Objects received: 1
B stores: {'sender': 'Alice', 'recipient': 'Bob', 'amount': 10000, 'tx_id': 'A8-RELAY-001'}

5. REPEATED INVENTORY
Objects requested again: 0

6. ONWARD RELAY
C received the same transaction: True

7. ALTERED OBJECT REJECTED
Relay object hash mismatch.

ALL RELAY SIMULATION CHECKS PASSED.


### Relay demonstration
The demonstration checks that:
- Node B learns peer addresses from node A and excludes its own address.
- Node A announces a transaction identifier through inv.
- Node B requests the missing object through getdata.
- Node B checks the advertised content hash before storing the transaction.
- Repeated inventory produces no additional download request.
- Node B relays the transaction onwards to node C.
- Altered data is rejected when its content hash differs from the announcement.

The relay simulation is separate from the REST API's ledger. It demonstrates
the requested message flow without implying live peer synchronisation.

In [16]:
from pathlib import Path

readme_path = Path("README.md")
readme_marker = "## Graduate extension: simulated peer relay"

readme_extension = """
## Graduate extension: simulated peer relay
network.py also contains RelayNode, an in-memory demonstration of:
- getaddr / addr-style peer address exchange;
- inv announcements of transaction content hashes;
- getdata requests for missing objects;
- object delivery, content-hash checks and onward relay.

Run notebook Cells 14–15 to reproduce the demonstration.
Repeated inventory does not request already stored objects.
Altered objects fail the advertised content-hash check.

This extension does not add REST endpoints or alter the API ledger.
It does not implement live sockets, Bitcoin wire messages, full transaction
validation or distributed consensus.
"""

readme_text = readme_path.read_text(encoding="utf-8")
readme_text = readme_text.split(readme_marker)[0].rstrip()

readme_path.write_text(
    readme_text + "\n\n" + readme_extension.strip() + "\n",
    encoding="utf-8"
)

print("README updated with the relay extension.")

README updated with the relay extension.


### Extension documentation
The README now documents the simulated networking extension and its limits.
The original REST endpoint contracts remain consistent with the implementation.

In [17]:
import json
from threading import Thread
from urllib.request import Request, urlopen
from werkzeug.serving import make_server

# A fresh application keeps this demonstration reproducible.
live_app = api.create_app()

# Port 0 selects an available local port.
server = make_server("127.0.0.1", 0, live_app)
port = server.server_port
base_url = f"http://127.0.0.1:{port}"

server_thread = Thread(target=server.serve_forever, daemon=True)
server_thread.start()


def http_request(method, path, payload=None):
    data = None
    headers = {}

    if payload is not None:
        data = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"

    request = Request(
        base_url + path,
        data=data,
        headers=headers,
        method=method
    )

    with urlopen(request, timeout=10) as response:
        raw = response.read().decode("utf-8")
        content_type = response.headers.get("Content-Type", "")

        body = (
            json.loads(raw)
            if "application/json" in content_type
            else raw
        )
        return response.status, body


try:
    print("LIVE LOCALHOST HTTP DEMONSTRATION")
    print("Server address:", base_url)

    status, initial = http_request("GET", "/chain")
    assert status == 200
    assert initial["length"] == 1
    print("\nInitial chain: HTTP", status, "| length:", initial["length"])

    live_payment = {
        "sender": "Alice",
        "recipient": "Bob",
        "amount": 10000,
        "tx_id": "A8-LIVE-001"
    }

    status, submitted = http_request(
        "POST", "/transactions/new", live_payment
    )
    assert status == 201
    assert submitted["status"] == "pending"
    print("Submit payment: HTTP", status, "| status:", submitted["status"])

    status, pending = http_request("GET", "/chain")
    assert pending["length"] == 1
    assert pending["pending_transactions"] == 1
    print("Before mining: chain length 1 | pending payments 1")

    status, retry = http_request(
        "POST", "/transactions/new", live_payment
    )
    assert status == 200
    assert retry["duplicate"] is True
    print("Identical retry: HTTP", status, "| duplicate:", retry["duplicate"])

    status, mined = http_request("POST", "/mine")
    assert status == 200
    assert any(
        tx["id"] == "A8-LIVE-001"
        for tx in mined["block"]["transactions"]
    )
    print("Mine: HTTP", status, "| block index:", mined["block"]["index"])

    status, final_chain = http_request("GET", "/chain")
    assert status == 200
    assert final_chain["length"] == 2
    assert final_chain["pending_transactions"] == 0
    assert final_chain["valid"] is True

    print(
        "Final chain: length =", final_chain["length"],
        "| pending =", final_chain["pending_transactions"],
        "| valid =", final_chain["valid"]
    )

    status, registered = http_request(
        "POST",
        "/nodes/register",
        {"nodes": ["http://127.0.0.1:5001/"]}
    )
    assert status == 201

    status, peers = http_request("GET", "/nodes")
    assert status == 200
    assert peers["nodes"] == ["http://127.0.0.1:5001"]
    print("Peer registration and listing: PASSED")

    status, explorer_page = http_request("GET", "/explorer")
    assert status == 200
    assert "A8-LIVE-001" in explorer_page
    assert "Alice" in explorer_page
    assert "Bob" in explorer_page
    print("Explorer over HTTP: PASSED")

    print("\nALL LIVE LOCALHOST HTTP CHECKS PASSED.")

finally:
    server.shutdown()
    server_thread.join(timeout=5)
    server.server_close()
    print("Temporary localhost server stopped.")

127.0.0.1 - - [09/Oct/2026 01:53:36] "GET /chain HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 01:53:36] "

LIVE LOCALHOST HTTP DEMONSTRATION
Server address: http://127.0.0.1:49487

Initial chain: HTTP 200 | length: 1


POST /transactions/new HTTP/1.1" 201 -
127.0.0.1 - - [09/Oct/2026 01:53:36] "GET /chain HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 01:53:36] "POST /transactions/new HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 01:53:36] "POST /mine HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 01:53:36] "GET /chain HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 01:53:36] "POST /nodes/register HTTP/1.1" 201 -
127.0.0.1 - - [09/Oct/2026 01:53:36] "GET /nodes HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 01:53:36] "GET /explorer HTTP/1.1" 200 -


Submit payment: HTTP 201 | status: pending
Before mining: chain length 1 | pending payments 1
Identical retry: HTTP 200 | duplicate: True
Mine: HTTP 200 | block index: 1
Final chain: length = 2 | pending = 0 | valid = True
Peer registration and listing: PASSED
Explorer over HTTP: PASSED

ALL LIVE LOCALHOST HTTP CHECKS PASSED.
Temporary localhost server stopped.


### Live localhost HTTP verification
This demonstration exercises the Flask API through actual HTTP requests to
a temporary server bound to 127.0.0.1. It checks submission, pending state,
idempotent retry, mining, chain verification, peer registration and the
HTML explorer.

The server uses an available local port and shuts down automatically.
Registered peer addresses remain directory entries; this test does not
establish connections to those peers or implement distributed consensus.